# Manual Review: UNSW Abstract Baseline Comparison



## 1. Import libraries and set paths


In [1]:
import os
import gc
import json
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed

SEED = 42
set_seed(SEED)
THESIS_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents]
                  if (path / "base-lines-sdgi-report-grouped").exists())
RUN_DIR = THESIS_DIR / "manual_review"
RESULTS_DIR = RUN_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_DIR = THESIS_DIR / "base-lines-sdgi-report-grouped"
MODEL_NAMES = ["xlm_roberta", "bert", "scibert", "biobert", "roberta_base", "deberta_v3"]
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE)


/Users/aman/micromamba/envs/thesis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## 2. Load the dataset

I used 43 texts: 8 showcase research summaries and 35 OpenAlex abstracts.


In [2]:
df = pd.read_csv(RUN_DIR / "dataset/unsw_sdg_abstracts.csv")
df["labels"] = df["labels"].apply(json.loads)
assert len(df) == 43 and df["text"].nunique() == 43
print("Abstracts:", len(df))
display(df.head())


Abstracts: 43


,text,labels
0,Waste is a resource waiting to be transformed ...,"[9, 12, 13]"
1,"Historically, research and advocacy in busines...","[8, 12, 16]"
2,The world is grappling with rising environment...,"[11, 13]"
3,"Water Security for Food, Livelihoods, and Clim...","[2, 6, 13, 17]"
4,This pilot research project explores the exper...,"[10, 11]"


### 2.1. Connect to W&B


In [ ]:
import wandb

WANDB_PROJECT = "thesis-manual-review"
WANDB_ENTITY = None
RUN_NAME = "unsw-43-baseline-comparison"

wandb.finish()
os.environ["WANDB_NOTEBOOK_NAME"] = str(RUN_DIR / "manual_review.ipynb")
with open(THESIS_DIR / ".env") as env_file:
    for line in env_file:
        name, separator, value = line.strip().removeprefix("export ").partition("=")
        if separator and name.strip() in ["WANDB_API_KEY", "HF_TOKEN"]:
            os.environ[name.strip()] = value.strip().strip("\"'")
if not os.environ.get("WANDB_API_KEY"):
    raise ValueError("WANDB_API_KEY is missing from the thesis .env file.")
wandb.login()
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=RUN_NAME,
    group="unsw-43",
    job_type="evaluation",
    dir=str(RUN_DIR),
    config={
        "dataset": "unsw_sdg_abstracts", "items": len(df), "seed": SEED,
        "batch_size": 4, "max_length": 512, "threshold": 0.5,
        "metrics_average": "samples", "device": DEVICE,
        "model_count": 12,
    },
)
print("W&B run:", run.url)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from WANDB_API_KEY.


wandb: Currently logged in as: amansharma0603 (amansharma0603-university-of-new-south-whales) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: setting up run aj3fj5re


wandb: Tracking run with wandb version 0.31.0


wandb: Run data is saved locally in /Users/aman/Desktop/Thesis/manual_review/wandb/run-20261102_225712-aj3fj5re
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run unsw-43-baseline-comparison


wandb: ⭐️ View project at https://forge.coreweave.com/wandb/amansharma0603-university-of-new-south-whales/thesis-manual-review


wandb: 🚀 View run at https://forge.coreweave.com/wandb/amansharma0603-university-of-new-south-whales/thesis-manual-review/runs/aj3fj5re


W&B run: https://forge.coreweave.com/wandb/amansharma0603-university-of-new-south-whales/thesis-manual-review/runs/aj3fj5re


## 3. Encode SDG labels and tokenize passages


In [4]:
true_labels = np.array([[int(goal in goals) for goal in range(1, 18)] for goals in df["labels"]])
texts = df["text"].tolist()

def tokenize(tokenizer, batch):
    return tokenizer(batch, truncation=True, max_length=512,
                     padding=True, return_tensors="pt")


## 4. Load the models and define metrics

I used the saved fine-tuned baseline models and tokenizers. Precision, recall and F1 are averaged across passages; accuracy requires an exact label-set match.


In [5]:
def load_model(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
    assert model.config.id2label == {index: f"SDG {index + 1}" for index in range(17)}
    model.eval()
    return tokenizer, model.to(DEVICE)

def compute_metrics(predicted_labels):
    return {
        "precision": precision_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "recall": recall_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "f1": f1_score(true_labels, predicted_labels, average="samples", zero_division=0),
    }


## 5. Evaluate all baselines

Evaluate=ing both all-language and English-only variants, loading one model at a time.


In [6]:
results = []
predictions = []
for group in ["all", "en"]:
    for model_name in MODEL_NAMES:
        baseline = f"{model_name}/{group}"
        tokenizer, model = load_model(MODEL_DIR / group / model_name / "model")
        batches = []
        with torch.inference_mode():
            for start in range(0, len(texts), 4):
                inputs = tokenize(tokenizer, texts[start:start + 4])
                inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
                batches.append(torch.sigmoid(model(**inputs).logits).cpu().numpy())
        predicted_labels = (np.concatenate(batches) >= 0.5).astype(int)
        scores = compute_metrics(predicted_labels)
        results.append({"baseline": baseline, "model": model_name, "training_variant": group, **scores})
        for index, labels in enumerate(predicted_labels):
            predictions.append({"baseline": baseline, "row_id": f"UNSW-{index + 1:03}",
                                "true_labels": json.dumps(df.iloc[index]["labels"]),
                                "predicted_labels": json.dumps((np.flatnonzero(labels) + 1).tolist())})
        run.log({f"{baseline}/{key}": value for key, value in scores.items()})
        print(f"{baseline}: F1 = {scores['f1']:.4f}")
        del model, tokenizer, inputs
        gc.collect()
        if DEVICE == "mps":
            torch.mps.empty_cache()
        elif DEVICE == "cuda":
            torch.cuda.empty_cache()


xlm_roberta/all: F1 = 0.6203


bert/all: F1 = 0.6240


scibert/all: F1 = 0.6973


biobert/all: F1 = 0.6754


roberta_base/all: F1 = 0.6633


/Users/aman/micromamba/envs/thesis/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


... (3 outputs left out)

scibert/en: F1 = 0.6792


biobert/en: F1 = 0.6426


roberta_base/en: F1 = 0.6442


deberta_v3/en: F1 = 0.6093


## 6. Compare results and save


In [7]:
results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
results_df.to_csv(RESULTS_DIR / "comparison.csv", index=False)
pd.DataFrame(predictions).to_csv(RESULTS_DIR / "predictions.csv", index=False)
display(results_df[["baseline", "precision", "recall", "f1", "accuracy"]])

run.log({"comparison": wandb.Table(dataframe=results_df)})
run.summary["best_baseline_by_samples_f1"] = results_df.iloc[0]["baseline"]
run.summary["best_samples_f1"] = float(results_df.iloc[0]["f1"])
run.summary["evaluated_baselines"] = len(results_df)
artifact = wandb.Artifact("unsw-43-evaluation", type="evaluation")
for path in [RUN_DIR / "dataset/unsw_sdg_abstracts.csv",
             RUN_DIR / "dataset/provenance.json",
             RESULTS_DIR / "comparison.csv", RESULTS_DIR / "predictions.csv"]:
    artifact.add_file(str(path))
run.log_artifact(artifact)
(RESULTS_DIR / "wandb_run.json").write_text(json.dumps(
    {"url": run.url, "id": run.id, "path": run.path}, indent=2))
print("W&B run:", run.url)
print("Results saved to:", RESULTS_DIR)
run.finish()


,baseline,precision,recall,f1,accuracy
0,scibert/all,0.709302,0.773256,0.697287,0.279070
1,scibert/en,0.833333,0.624031,0.679181,0.255814
2,biobert/all,0.697674,0.757752,0.675360,0.209302
3,bert/en,0.782946,0.641473,0.668992,0.279070
4,roberta_base/all,0.813953,0.645349,0.663344,0.232558
5,xlm_roberta/en,0.770930,0.649225,0.662348,0.302326
6,roberta_base/en,0.781008,0.610465,0.644186,0.255814
7,biobert/en,0.833333,0.563953,0.642636,0.186047
8,bert/all,0.763566,0.569767,0.624031,0.255814
9,xlm_roberta/all,0.748062,0.581395,0.620266,0.209302


wandb: uploading artifact run-aj3fj5re-comparison; updating run metadata; uploading artifact unsw-43-evaluation


W&B run: https://forge.coreweave.com/wandb/amansharma0603-university-of-new-south-whales/thesis-manual-review/runs/aj3fj5re
Results saved to: /Users/aman/Desktop/Thesis/manual_review/results


wandb: uploading artifact run-aj3fj5re-comparison; uploading artifact unsw-43-evaluation


wandb: uploading artifact unsw-43-evaluation


wandb: uploading media/table/comparison_12_2750516c4f5d5b17e96b.table.json; uploading output.log; uploading wandb-summary.json


wandb: uploading media/table/comparison_12_2750516c4f5d5b17e96b.table.json


wandb: 
wandb: Run history:
wandb:    bert/all/accuracy ▁
wandb:          bert/all/f1 ▁
wandb:   bert/all/precision ▁
wandb:      bert/all/recall ▁
wandb:     bert/en/accuracy ▁
wandb:           bert/en/f1 ▁
wandb:    bert/en/precision ▁
wandb:       bert/en/recall ▁
wandb: biobert/all/accuracy ▁
wandb:       biobert/all/f1 ▁
wandb:                  +38 ...
wandb: 
wandb: Run summary:
wandb:           bert/all/accuracy 0.25581
wandb:                 bert/all/f1 0.62403
wandb:          bert/all/precision 0.76357
wandb:             bert/all/recall 0.56977
wandb:            bert/en/accuracy 0.27907
wandb:                  bert/en/f1 0.66899
wandb:           bert/en/precision 0.78295
wandb:              bert/en/recall 0.64147
wandb: best_baseline_by_samples_f1 scibert/all
wandb:             best_samples_f1 0.69729
wandb:                         +41 ...
wandb: 


wandb: 🚀 View run unsw-43-baseline-comparison at: https://forge.coreweave.com/wandb/amansharma0603-university-of-new-south-whales/thesis-manual-review/runs/aj3fj5re
wandb: ⭐️ View project at: https://forge.coreweave.com/wandb/amansharma0603-university-of-new-south-whales/thesis-manual-review
wandb: Synced 5 W&B file(s), 1 media file(s), 4 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261102_225712-aj3fj5re/logs
